# 03 · Camada Gold — Data Mart Analítico (L)

Este é o notebook final do pipeline, responsável por construir o Modelo Dimensional (*Star Schema*) que será consumido diretamente pela equipa de negócio através de ferramentas de BI (Power BI). Representa o **"L" (Load)** do processo ELT.

**O que faz este notebook:**
1. **Geração do Calendário:** Cria programaticamente uma dimensão de tempo (`Dim_Date`) contínua para suportar todas as análises temporais.
2. **Geração de Surrogate Keys:** Transforma as *Business Keys* originais em chaves artificiais (inteiros) ultrarrápidas para o cruzamento de tabelas.
3. **Carga de Dimensões:** Popula as tabelas de contexto de negócio (Cliente, Produto, Geografia, Vendedor).
4. **Carga de Factos:** Agrega as métricas financeiras (Vendas, Encomendas e Transações) ao seu nível de granularidade mais baixo, garantindo as ligações corretas à cidade de entrega (`deliverycityid`).
5. **Otimização Analítica:** Cria os índices B-Tree necessários (ex: `idx_fs_date`) para garantir performance máxima na visualização dos *dashboards*.



### 1. Inicialização e Ligações
Importação de dependências essenciais e estabelecimento da ligação ao nosso Data Warehouse (`wwi_dw`). Esta é a etapa de preparação para a carga final dos dados limpos no nosso modelo dimensional.

In [1]:
import pandas as pd
from datetime import datetime, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

# Conexão ao DW
engine_dwh = create_engine(
    f"postgresql+psycopg2://{os.getenv('DWH_USER')}:{os.getenv('DWH_PASSWORD')}@"
    f"{os.getenv('DWH_HOST')}:{os.getenv('DWH_PORT')}/{os.getenv('DWH_DB')}"
)

run_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"✓ Motores prontos. Início da Carga Gold: {run_at}")

✓ Motores prontos. Início da Carga Gold: 2026-04-19 20:31:10.340894


### 2. Geração do Calendário (Dim_Date)
A base de dados transacional da WWI não possui uma tabela de datas explícita. Para suportar as análises temporais nos nossos dashboards, geramos aqui programaticamente uma dimensão de tempo contínua (2017-2020), extraindo atributos como o Ano, Trimestre e Mês. O resultado final inclui a criação de uma *Surrogate Key* em formato numérico (ex: `20170101`).

In [2]:
# Criar intervalo de datas
dates = pd.date_range(start="2017-01-01", end="2020-12-31")

df_date = pd.DataFrame({"fulldate": dates})
df_date["datekey"]   = df_date["fulldate"].dt.strftime('%Y%m%d').astype(int)
df_date["year"]      = df_date["fulldate"].dt.year
df_date["quarter"]   = df_date["fulldate"].dt.quarter
df_date["month"]     = df_date["fulldate"].dt.month
df_date["monthname"] = df_date["fulldate"].dt.month_name()

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.dim_date CASCADE;"))
    df_date.to_sql("dim_date", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Dim_Date carregada com {len(df_date)} dias.")

✓ Dim_Date carregada com 1461 dias.


### 3. Carregar Dimensões (SCD Tipo 1)
Carga das tabelas dimensionais que não requerem histórico de alterações: `Dim_Geography`, `Dim_Customer` e `Dim_Salesperson`. A abordagem *Slowly Changing Dimension Tipo 1* significa que apenas o estado atual de cada registo é preservado no nosso modelo, garantindo queries analíticas simples e rápidas.

In [3]:
with engine_dwh.begin() as conn:
    # 1. Geografia
    conn.execute(text("TRUNCATE TABLE gold.dim_geography CASCADE;"))
    df_geo = pd.read_sql("SELECT cityid, cityname, stateprovincename, countryname, latestrecordedpopulation FROM silver.stg_geography", conn)
    df_geo.to_sql("dim_geography", conn, schema="gold", if_exists="append", index=False)

    # 2. Clientes
    conn.execute(text("TRUNCATE TABLE gold.dim_customer CASCADE;"))
    df_cust = pd.read_sql("SELECT customerid, customername, buyinggroupname, customercategoryname, creditlimit, accountopeneddate, paymentdays FROM silver.stg_customers", conn)
    df_cust.to_sql("dim_customer", conn, schema="gold", if_exists="append", index=False)

    # 3. Vendedores
    conn.execute(text("TRUNCATE TABLE gold.dim_salesperson CASCADE;"))
    df_sales = pd.read_sql("SELECT salespersonid, fullname, preferredname, phonenumber FROM silver.stg_salespersons", conn)
    df_sales.to_sql("dim_salesperson", conn, schema="gold", if_exists="append", index=False)

print("✓ Dimensões SCD1 (Customer, Salesperson, Geography) carregadas.")

✓ Dimensões SCD1 (Customer, Salesperson, Geography) carregadas.


### 4. Carregar Dimensão de Produtos (SCD Tipo 2)
Carga da `Dim_StockItem`. Ao contrário das dimensões anteriores, esta tabela importa diretamente a estrutura com histórico que construímos na camada Silver (`dim_stockitems_scd2`). Isto garante que qualquer análise associará uma venda passada à versão exata que o produto tinha nesse momento específico.

In [4]:
with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.dim_stockitem CASCADE;"))
    sql_stock = """
        SELECT stockitemid, stockitemname, brand, colorname, size, leadtimedays, 
               quantityperouter, stockgroupname, valid_from as date_from, 
               valid_to as date_to, is_current, scd_key as version
        FROM silver.dim_stockitems_scd2
    """
    df_stock = pd.read_sql(sql_stock, conn)
    df_stock.to_sql("dim_stockitem", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Dim_StockItem (SCD2) carregada com {len(df_stock)} registos históricos.")

✓ Dim_StockItem (SCD2) carregada com 227 registos históricos.


### 5. Carregar Tabela de Factos de Vendas (Fact_Sales)
Tabela central do nosso *Star Schema*. Esta tabela de factos agrega a faturação da WWI ao seu nível mais detalhado (linha de fatura). Aqui efetuamos os JOINs finais para substituir as *Business Keys* originais pelas nossas *Surrogate Keys* (chaves artificiais), assegurando que a dimensão geográfica é cruzada corretamente com a Cidade de Entrega (`deliverycityid`).

In [5]:
sql_fact_sales = """
    SELECT 
        il.invoicelineid,
        CAST(TO_CHAR(il.invoicedate, 'YYYYMMDD') AS INT) as datekey,
        c.customerkey,
        s.stockitemkey,
        g.geographykey,
        sp.salespersonkey,
        il.quantity,
        il.extendedprice,
        il.lineprofit,
        il.taxamount
    FROM silver.stg_invoicelines il
    JOIN silver.stg_invoices i ON i.invoiceid = il.invoiceid
    JOIN silver.stg_customers sc ON sc.customerid = i.customerid   -- < Buscar a info do cliente à Silver
    JOIN gold.dim_customer c ON c.customerid = i.customerid
    JOIN gold.dim_stockitem s ON s.stockitemid = il.stockitemid AND s.is_current = TRUE
    JOIN gold.dim_geography g ON g.cityid = sc.deliverycityid      -- < Agora sim, a Cidade de Entrega!
    JOIN gold.dim_salesperson sp ON sp.salespersonid = i.salespersonpersonid
"""

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_sales CASCADE;"))
    df_fact = pd.read_sql(sql_fact_sales, conn)
    df_fact.to_sql("fact_sales", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Fact_Sales carregada: {len(df_fact)} linhas de venda (Geografia Corrigida!).")

✓ Fact_Sales carregada: 228265 linhas de venda (Geografia Corrigida!).


### 6. Carregar Tabela de Factos de Encomendas (Fact_Orders)
Tabela dedicada à monitorização da procura. Ao registarmos a facturação ao nível da linha de encomenda, a WWI passa a conseguir analisar as quantidades solicitadas pelos clientes, mesmo que estas sofram alterações logísticas antes de se transformarem em vendas efetivas.

In [6]:
sql_fact_orders = """
    SELECT 
        ol.orderlineid,
        CAST(TO_CHAR(ol.orderdate, 'YYYYMMDD') AS INT) as datekey,
        c.customerkey,
        s.stockitemkey,
        g.geographykey,
        sp.salespersonkey,
        ol.quantity,
        ol.unitprice,
        ol.taxrate
    FROM silver.stg_orderlines ol
    JOIN silver.stg_orders o ON o.orderid = ol.orderid
    JOIN silver.stg_customers sc ON sc.customerid = o.customerid   -- < Buscar a info do cliente à Silver
    JOIN gold.dim_customer c ON c.customerid = o.customerid
    JOIN gold.dim_stockitem s ON s.stockitemid = ol.stockitemid AND s.is_current = TRUE
    JOIN gold.dim_geography g ON g.cityid = sc.deliverycityid      -- < Ligar à Cidade Correta
    JOIN gold.dim_salesperson sp ON sp.salespersonid = o.salespersonpersonid
"""

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_orders CASCADE;"))
    df_fact_orders = pd.read_sql(sql_fact_orders, conn)
    df_fact_orders.to_sql("fact_orders", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Fact_Orders carregada: {len(df_fact_orders)} linhas de encomenda (Geografia Corrigida!).")

✓ Fact_Orders carregada: 231412 linhas de encomenda (Geografia Corrigida!).


### 7. Carregar Tabela de Factos Transações Financeiras (Fact_CustomerTransactions)
Processo de negócio crucial para monitorizar a saúde financeira e o risco de crédito da empresa. Esta tabela carrega as transações bancárias e faturas emitidas, mapeando a métrica vital de dívida pendente (`OutstandingBalance`) do cliente no momento da operação.

In [7]:
sql_fact_transactions = """
    SELECT 
        ct.customertransactionid,
        CAST(TO_CHAR(ct.transactiondate, 'YYYYMMDD') AS INT) as datekey,
        c.customerkey,
        ct.transactionamount,
        ct.outstandingbalance,
        ct.isfinalized
    FROM silver.stg_customertransactions ct
    JOIN gold.dim_customer c ON c.customerid = ct.customerid
"""

with engine_dwh.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.fact_customertransactions CASCADE;"))
    df_fact_ct = pd.read_sql(sql_fact_transactions, conn)
    df_fact_ct.to_sql("fact_customertransactions", conn, schema="gold", if_exists="append", index=False)

print(f"✓ Fact_CustomerTransactions carregada: {len(df_fact_ct)} transações financeiras.")

✓ Fact_CustomerTransactions carregada: 97147 transações financeiras.


### 8. Validação Final do Data Mart
Auditoria de encerramento do processo. Este sumário lista a contagem total de linhas carregadas em cada uma das tabelas da camada Gold, permitindo-nos confirmar que todos os dados transacionais e dimensionais transitaram com sucesso até ao final do pipeline.

In [8]:
print(f"{'Tabela Gold':<30} {'Linhas':>10}")
print("-" * 42)

gold_tables = [
    "dim_date", "dim_customer", "dim_stockitem", "dim_geography", 
    "dim_salesperson", "fact_sales", "fact_orders", "fact_customertransactions"
]

for table in gold_tables:
    with engine_dwh.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM gold.{table}")).scalar()
        print(f"{table:<30} {n:>10}")

Tabela Gold                        Linhas
------------------------------------------
dim_date                             1461
dim_customer                          663
dim_stockitem                         227
dim_geography                       37940
dim_salesperson                        10
fact_sales                         228265
fact_orders                        231412
fact_customertransactions           97147


## 9. Resumo da Camada Gold

A construção da camada Gold marca a conclusão do processo **ELT (Extract, Load, Transform)** e a estabilização da *Medallion Architecture*.

O **Data Mart da WWI** está formalmente desenhado num Esquema em Estrela (*Star Schema*), centralizado em 3 Tabelas de Factos que respondem a processos de negócio distintos. Os dados estão limpos, cruzados e indexados de forma a maximizar a performance. 

A partir deste momento, o Data Warehouse local encontra-se pronto para ser ligado e consumido por ferramentas de *Business Intelligence* (como o Power BI) para a extração direta de insights para gestão!